# Semana 3: Representación del Texto: BOW, TF-IDF y N-gramas
## Notebook Conceptual (NB1) – Construcción Manual de Vectores

**Propósito:** Transformar texto en vectores numéricos utilizando técnicas clásicas y entender sus limitaciones.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Construir manualmente el vocabulario y la matriz BOW (Bag of Words).
- Calcular TF-IDF paso a paso: frecuencia en documento, frecuencia inversa.
- Comparar vectores BOW y TF-IDF y discutir qué palabras se ponderan más.
- Generar n-gramas (bigramas, trigramas) y observar cómo capturan contexto.
- Usar CountVectorizer y TfidfVectorizer de sklearn y verificar resultados.

---

## 0. Configuración Inicial

Importamos las librerías necesarias.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Librerías importadas correctamente.")

---
## 1. Corpus Dummy

Definimos tres oraciones muy cortas para trabajar manualmente.

In [ ]:
corpus = [
    "el gato come pescado",
    "el perro come carne",
    "el gato come carne"
]

print("=== CORPUS ===")
for i, doc in enumerate(corpus):
    print(f"d{i+1}: {doc}")

---
## 2. Construcción Manual de Bag of Words (BOW)

### 2.1. Creación del vocabulario

Extraemos todas las palabras únicas del corpus y las ordenamos.

In [ ]:
# Tokenización simple por espacios
tokens_list = [doc.split() for doc in corpus]
print("Tokens por documento:")
for i, tokens in enumerate(tokens_list):
    print(f"d{i+1}: {tokens}")

# Crear vocabulario (palabras únicas ordenadas)
vocabulario = sorted(set([word for tokens in tokens_list for word in tokens]))
print(f"\nVocabulario ({len(vocabulario)} palabras):")
print(vocabulario)

### 2.2. Matriz BOW (frecuencias)

Construimos la matriz documento-término donde cada fila representa un documento y cada columna una palabra del vocabulario, con el número de ocurrencias.

In [ ]:
# Inicializar matriz de ceros
bow_matrix = np.zeros((len(corpus), len(vocabulario)), dtype=int)

# Llenar la matriz
for i, tokens in enumerate(tokens_list):
    for word in tokens:
        j = vocabulario.index(word)
        bow_matrix[i, j] += 1

# Mostrar como DataFrame
df_bow = pd.DataFrame(bow_matrix, columns=vocabulario, index=[f'd{i+1}' for i in range(len(corpus))])
print("=== Matriz BOW (frecuencias) ===")
df_bow

### 2.3. Verificación con CountVectorizer de sklearn

In [ ]:
# Usar CountVectorizer
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(corpus)

# Obtener vocabulario y matriz
vocab_sklearn = count_vectorizer.get_feature_names_out()
bow_sklearn = X_count.toarray()

df_bow_sklearn = pd.DataFrame(bow_sklearn, columns=vocab_sklearn, index=[f'd{i+1}' for i in range(len(corpus))])
print("=== Matriz BOW (sklearn CountVectorizer) ===")
df_bow_sklearn

# Verificar que coinciden
print("\n¿Coinciden las matrices?", np.array_equal(bow_matrix, bow_sklearn))
if not np.array_equal(bow_matrix, bow_sklearn):
    print("¡Atención! Puede haber diferencias en el orden del vocabulario.")

---
## 3. Cálculo Manual de TF-IDF

### 3.1. Frecuencia de término (TF)

Usaremos la frecuencia bruta (raw count) como TF.

In [ ]:
# TF es la matriz BOW que ya tenemos
tf = bow_matrix.copy()
print("TF (frecuencia de término):")
pd.DataFrame(tf, columns=vocabulario, index=[f'd{i+1}' for i in range(len(corpus))])

### 3.2. Frecuencia inversa de documento (IDF)

Calculamos cuántos documentos contienen cada término (document frequency).

In [ ]:
N = len(corpus)  # número total de documentos

# Document frequency: número de documentos que contienen cada palabra
df = np.zeros(len(vocabulario))
for j in range(len(vocabulario)):
    df[j] = np.sum(bow_matrix[:, j] > 0)

print("Document frequency (df):")
for word, freq in zip(vocabulario, df):
    print(f"{word:10} -> aparece en {int(freq)} documento(s)")

# Calcular IDF
idf = np.log(N / (df + 1e-10))  # +1e-10 para evitar división por cero
idf = np.round(idf, 4)

print("\nIDF (log(N/df)):")
for word, value in zip(vocabulario, idf):
    print(f"{word:10} -> {value:.4f}")

### 3.3. Matriz TF-IDF (sin normalizar)

In [ ]:
tfidf_matrix = tf * idf  # multiplicación elemento a elemento
tfidf_matrix = np.round(tfidf_matrix, 4)

df_tfidf = pd.DataFrame(tfidf_matrix, columns=vocabulario, index=[f'd{i+1}' for i in range(len(corpus))])
print("=== Matriz TF-IDF (sin normalizar) ===")
df_tfidf

### 3.4. Verificación con TfidfVectorizer de sklearn

Nota: sklearn usa una fórmula ligeramente diferente (IDF = log((N+1)/(df+1)) + 1 y luego normaliza L2).

In [ ]:
# Usar TfidfVectorizer con configuración estándar
tfidf_vectorizer = TfidfVectorizer(norm=None, smooth_idf=False, use_idf=True)
X_tfidf = tfidf_vectorizer.fit_transform(corpus)
tfidf_sklearn = X_tfidf.toarray()

vocab_tfidf = tfidf_vectorizer.get_feature_names_out()
df_tfidf_sklearn = pd.DataFrame(tfidf_sklearn, columns=vocab_tfidf, index=[f'd{i+1}' for i in range(len(corpus))])

print("=== Matriz TF-IDF (sklearn, sin normalizar) ===")
df_tfidf_sklearn.round(4)

# Ver IDF calculado por sklearn
print("\nIDF según sklearn:")
idf_sklearn = tfidf_vectorizer.idf_
for word, value in zip(vocab_tfidf, idf_sklearn):
    print(f"{word:10} -> {value:.4f}")

---
## 4. Comparación BOW vs TF-IDF

Analizamos qué palabras tienen mayor peso en cada representación.

In [ ]:
print("=== Comparación de pesos por palabra ===\n")

for i, doc_name in enumerate([f'd{i+1}' for i in range(len(corpus))]):
    print(f"Documento {doc_name}: '{corpus[i]}'")
    print("  BOW (frecuencia):")
    for j, word in enumerate(vocabulario):
        if bow_matrix[i, j] > 0:
            print(f"    {word}: {bow_matrix[i, j]}")
    print("  TF-IDF:")
    for j, word in enumerate(vocabulario):
        if tfidf_matrix[i, j] > 0:
            print(f"    {word}: {tfidf_matrix[i, j]:.4f}")
    print()

### Discusión:

1. **¿Qué palabras tienen mayor peso en BOW?**
   - Todas las palabras tienen peso proporcional a su frecuencia. 'el' aparece en todos los documentos con frecuencia 1.

2. **¿Qué palabras tienen mayor peso en TF-IDF?**
   - Las palabras que aparecen en pocos documentos ('pescado', 'perro', 'carne') tienen mayor IDF y por tanto mayor peso TF-IDF.
   - 'el' tiene IDF = 0 porque aparece en todos los documentos, por lo que su contribución es nula.

3. **¿Por qué TF-IDF es preferible para tareas como búsqueda de documentos similares?**
   - Porque reduce el ruido de palabras comunes y resalta las palabras distintivas.

---
## 5. N-gramas

Generamos bigramas y trigramas para ver cómo capturan contexto.

In [ ]:
def generate_ngrams(tokens, n):
    """Genera n-gramas a partir de una lista de tokens."""
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# Mostrar bigramas para cada documento
print("=== Bigramas ===")
for i, tokens in enumerate(tokens_list):
    bigrams = generate_ngrams(tokens, 2)
    print(f"d{i+1}: {bigrams}")

print("\n=== Trigramas ===")
for i, tokens in enumerate(tokens_list):
    trigrams = generate_ngrams(tokens, 3)
    print(f"d{i+1}: {trigrams}")

### 5.1. N-gramas con CountVectorizer

In [ ]:
# Bigramas
bigram_vectorizer = CountVectorizer(ngram_range=(2,2))
X_bigrams = bigram_vectorizer.fit_transform(corpus)

print("=== Bigramas con CountVectorizer ===")
print("Vocabulario de bigramas:", bigram_vectorizer.get_feature_names_out())
print("Matriz de bigramas:")
print(X_bigrams.toarray())

# Trigramas
trigram_vectorizer = CountVectorizer(ngram_range=(3,3))
X_trigrams = trigram_vectorizer.fit_transform(corpus)

print("\n=== Trigramas con CountVectorizer ===")
print("Vocabulario de trigramas:", trigram_vectorizer.get_feature_names_out())
print("Matriz de trigramas:")
print(X_trigrams.toarray())

### 5.2. Combinación de unigramas y bigramas

In [ ]:
combined_vectorizer = CountVectorizer(ngram_range=(1,2))
X_combined = combined_vectorizer.fit_transform(corpus)

print("=== Unigramas + Bigramas ===")
print("Vocabulario:", combined_vectorizer.get_feature_names_out())
print("Matriz:")
df_combined = pd.DataFrame(X_combined.toarray(), 
                           columns=combined_vectorizer.get_feature_names_out(),
                           index=[f'd{i+1}' for i in range(len(corpus))])
df_combined

---
## 6. Similitud entre documentos

Calculamos la similitud del coseno entre documentos usando las representaciones BOW y TF-IDF.

In [ ]:
# Similitud con BOW
cos_sim_bow = cosine_similarity(bow_matrix)
df_cos_bow = pd.DataFrame(cos_sim_bow, index=[f'd{i+1}' for i in range(len(corpus))], 
                          columns=[f'd{i+1}' for i in range(len(corpus))])
print("=== Similitud del coseno con BOW ===")
df_cos_bow.round(4)

# Similitud con TF-IDF (normalizado)
# Normalizamos L2 nuestra matriz TF-IDF manual
tfidf_norm = tfidf_matrix / np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
cos_sim_tfidf = cosine_similarity(tfidf_norm)
df_cos_tfidf = pd.DataFrame(cos_sim_tfidf, index=[f'd{i+1}' for i in range(len(corpus))], 
                            columns=[f'd{i+1}' for i in range(len(corpus))])
print("\n=== Similitud del coseno con TF-IDF (normalizado) ===")
df_cos_tfidf.round(4)

### Interpretación de la similitud:

- d1 ("el gato come pescado") y d2 ("el perro come carne") comparten "el" y "come", por lo que tienen cierta similitud.
- d1 y d3 ("el gato come carne") comparten "el", "gato", "come", por lo que son más similares entre sí.
- d2 y d3 comparten "el", "come", "carne", también alta similitud.

En TF-IDF, la contribución de "el" se reduce, por lo que las similitudes se basan más en las palabras distintivas.

---
## 7. Ejercicio para el Estudiante

1. Añade una cuarta oración al corpus: "el pescado come gato" (una frase sin sentido pero interesante para ver cómo cambian los vectores).
2. Recalcula manualmente la matriz BOW y TF-IDF.
3. Observa cómo cambian las similitudes entre documentos.
4. ¿Qué palabras ganan o pierden importancia?

In [ ]:
# Espacio para el estudiante
# corpus_ampliado = corpus + ["el pescado come gato"]
# ...

---
## 8. Conclusiones

En este notebook hemos explorado las representaciones clásicas de texto:

✔️ **Bag of Words (BOW)**:
- Representación simple basada en frecuencias.
- Ignora el orden de las palabras.
- Las palabras comunes dominan.

✔️ **TF-IDF**:
- Pondera las palabras según su importancia en el documento y rareza en el corpus.
- Reduce el impacto de stopwords.
- Mejor para tareas de búsqueda y clasificación.

✔️ **N-gramas**:
- Capturan contexto local.
- Aumentan la dimensionalidad.
- Útiles cuando el orden importa.

✔️ **Verificación con sklearn**:
- CountVectorizer y TfidfVectorizer implementan estas técnicas eficientemente.
- Confirmamos que nuestros cálculos manuales coinciden (considerando diferencias en fórmulas).

**Lección clave**: La elección de la representación depende de la tarea. BOW y TF-IDF son puntos de partida sólidos y aún hoy se usan en muchos sistemas, aunque modelos modernos como embeddings los superan en tareas semánticas.

En el próximo notebook (NB2) aplicaremos estas técnicas a un dataset real de clasificación de noticias.

---
**Fin del Notebook Conceptual - Semana 3 (NLP)**